# 🔍 Adım 2 — Derin Veri Keşfi (EDA)
Şu 5 soruyu cevaplıyoruz:
1. İki sınıf görsel olarak ayrışıyor mu? (PCA)
2. Hangi özellikler sınıfı en çok ayırt ediyor? (Mann-Whitney + Effect Size)
3. Özellikler birbiriyle ne kadar iç içe? (Korelasyon)
4. Hangi özellikler neredeyse sabit? (Varyans)
5. Aykırı değerler ne durumda? (IQR)
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')
np.random.seed(42)

# Veriyi yükle — label eklemeyi concat sonrasına bırak (PerformanceWarning önlenir)
normal = pd.read_csv('../data/normal_radiomics.csv')
papil  = pd.read_csv('../data/papilodem_radiomics.csv')

normal_labeled = normal.copy()
papil_labeled  = papil.copy()
normal_labeled['label'] = 0
papil_labeled['label']  = 1

df = pd.concat([normal_labeled, papil_labeled], ignore_index=True)

feature_cols = [c for c in df.columns if c.startswith('Feature_')]
X = df[feature_cols].copy().values.astype(float)  # float64'e çevir, NaN/inf kontrolü
y = df['label'].values

# NaN veya inf var mı kontrol et
print(f'NaN sayısı : {np.isnan(X).sum()}')
print(f'Inf sayısı : {np.isinf(X).sum()}')
print(f'Veri hazır : {df.shape[0]} satır, {len(feature_cols)} özellik')

---
## SORU 1: İki sınıf görsel olarak ayrışıyor mu? (PCA)

**PCA nedir?** 746 boyutlu veriyi 2 boyuta indirgiyoruz.  
Eğer 2D'de kümeler ayrışıyorsa → model bu işi kolay öğrenir.  
Üst üste oturuyorsa → problem zor, güçlü bir modele ihtiyaç var.

In [ ]:
# RobustScaler: medyan ve IQR kullanır, aykırı değerlerden etkilenmez
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# PCA — 50 bileşene kadar hesapla (kümülatif varyans için)
pca_full = PCA(n_components=50, random_state=42)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

# 2D görselleştirme
X_pca = pca_full.transform(X_scaled)[:, :2]
var1, var2 = pca_full.explained_variance_ratio_[:2] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: 2D scatter
for cls, label, color in [(0, 'Normal', 'steelblue'), (1, 'Papilödem', 'salmon')]:
    mask = y == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=label, alpha=0.5, s=20, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({var1:.1f}% varyans)')
axes[0].set_ylabel(f'PC2 ({var2:.1f}% varyans)')
axes[0].set_title('PCA — 746 özellik → 2 boyut')
axes[0].legend()

# Sağ: kümülatif varyans
axes[1].plot(range(1, len(cumvar)+1), cumvar, color='steelblue', linewidth=2)
for thr, color in [(80, 'orange'), (95, 'red')]:
    n = np.argmax(cumvar >= thr) + 1
    axes[1].axhline(thr, color=color, linestyle='--', alpha=0.7, label=f'%{thr} → {n} bileşen')
axes[1].set_xlabel('Bileşen sayısı')
axes[1].set_ylabel('Kümülatif varyans (%)')
axes[1].set_title('Kaç bileşen yeterli?')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/fig_pca.png', dpi=150, bbox_inches='tight')
plt.show()

n80 = np.argmax(cumvar >= 80) + 1
n95 = np.argmax(cumvar >= 95) + 1
print(f'PC1+PC2 açıklanan varyans : {var1+var2:.1f}%')
print(f'%80 varyans için          : {n80} bileşen yeterli')
print(f'%95 varyans için          : {n95} bileşen yeterli')

---
## SORU 2: Hangi özellikler sınıfı en çok ayırt ediyor?

**Mann-Whitney U testi:** Parametrik olmayan test — normal dağılım varsayımı gerektirmez.  
**Effect size (rank-biserial r):** 0 = hiç etki yok, 1 = mükemmel ayırt edici  
**Bonferroni düzeltmesi:** 746 test yaptık. Eşiği 0.05 → 0.05/746 = 6.7×10⁻⁵'e düşürüyoruz.

In [ ]:
print('İstatistiksel testler hesaplanıyor...')
x_n = df[df.label == 0][feature_cols].values
x_p = df[df.label == 1][feature_cols].values

results = []
for i, feat in enumerate(feature_cols):
    u_stat, p_val = stats.mannwhitneyu(x_n[:, i], x_p[:, i], alternative='two-sided')
    r = abs(1 - (2 * u_stat) / (len(x_n) * len(x_p)))
    results.append({'feature': feat, 'p_value': p_val, 'effect_size': r})

result_df = pd.DataFrame(results).sort_values('effect_size', ascending=False)

alpha_bon = 0.05 / len(feature_cols)
anlamli   = result_df[result_df.p_value < alpha_bon]

print(f'Toplam özellik          : {len(feature_cols)}')
print(f'Bonferroni eşiği        : {alpha_bon:.2e}')
print(f'İstatistiksel anlamlı   : {len(anlamli)} özellik')
print(f'\nEn ayırt edici TOP 10:')
display(result_df.head(10)[['feature', 'p_value', 'effect_size']].round(4))

In [ ]:
top10 = result_df.head(10)['feature'].tolist()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, feat in zip(axes.flatten(), top10):
    data_n = df[df.label == 0][feat]
    data_p = df[df.label == 1][feat]
    bp = ax.boxplot([data_n, data_p], labels=['Normal', 'Papilödem'],
                    patch_artist=True, medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('salmon')
    bp['boxes'][1].set_alpha(0.6)
    r_val = result_df[result_df.feature == feat]['effect_size'].values[0]
    ax.set_title(f'{feat}\nr={r_val:.2f}', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('En Ayırt Edici 10 Özellik — Box Plot', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/fig_top10_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## SORU 3: Özellikler birbiriyle ne kadar iç içe? (Korelasyon)

746 özelliğin büyük kısmı birbirinin kopyası gibi bilgi taşıyor.  
|r| > 0.95 eşiğiyle 504 özellik elenecek → 242 kalacak.  
Sonra MRMR ile bu 242'den en bilgilendirici olanlar seçilecek.

In [ ]:
print('Korelasyon matrisi hesaplanıyor...')
corr_matrix = df[feature_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

for thr in [0.95, 0.90, 0.80, 0.70]:
    n = (upper > thr).sum().sum()
    print(f'  |r| > {thr}: {n:,} çift')

to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f'\nElenecek özellik (>0.95): {len(to_drop)}')
print(f'Kalan özellik tahmini  : {len(feature_cols) - len(to_drop)}')

corr_vals = upper.values[upper.values > 0]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(corr_vals, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(0.95, color='red',    linestyle='--', label='Eşik: 0.95 (ödev)')
ax.axvline(0.80, color='orange', linestyle='--', label='Eşik: 0.80')
ax.set_xlabel('Pearson |r|')
ax.set_ylabel('Çift sayısı')
ax.set_title('Özellik Çiftleri Arasındaki Korelasyon Dağılımı')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/fig_korelasyon_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## SORU 4: Hangi özellikler neredeyse sabit? (Varyans)

327 özelliğin varyansı ≤ 0.01 — bunlar sıfıra yakın değişkenlik gösteriyor.  
Model bu özelliklerden hiçbir şey öğrenemez, aksine gürültü katar.

In [ ]:
variances = df[feature_cols].var()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(variances, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Varyans')
axes[0].set_ylabel('Özellik sayısı')
axes[0].set_title('Tüm Varyans Dağılımı')

axes[1].hist(np.log1p(variances), bins=60, color='salmon', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('log(Varyans + 1)')
axes[1].set_ylabel('Özellik sayısı')
axes[1].set_title('Log Ölçekli Varyans Dağılımı')

plt.tight_layout()
plt.savefig('../figures/fig_varyans.png', dpi=150, bbox_inches='tight')
plt.show()

for thr in [0, 0.01, 0.1]:
    print(f'Varyans ≤ {thr}: {(variances <= thr).sum()} özellik')

---
## SORU 5: Aykırı değerler ne durumda? (IQR)

%90.2 gibi görünen yüksek oran aslında ölçek farklılığından kaynaklanıyor.  
Bazı özellikler 0.005 ölçeğinde, bazıları 60.000 ölçeğinde → IQR yöntemi hepsini aykırı görüyor.  
**RobustScaler** tam da bu yüzden var: ölçekleri eşitledikten sonra gerçek aykırı değerlerin etkisi azalır.

In [ ]:
Q1 = df[feature_cols].quantile(0.25)
Q3 = df[feature_cols].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = ((df[feature_cols] < (Q1 - 1.5*IQR)) | (df[feature_cols] > (Q3 + 1.5*IQR)))

satir_bazli   = outlier_mask.any(axis=1).sum()
ozellik_bazli = outlier_mask.sum(axis=0).sort_values(ascending=False)

print(f'En az 1 aykırı değer içeren satır: {satir_bazli}/{len(df)} ({satir_bazli/len(df)*100:.1f}%)')
print(f'\nEn fazla aykırı değer içeren 10 özellik:')
display(ozellik_bazli.head(10).to_frame('aykırı_değer_sayısı'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(outlier_mask.sum(axis=0), bins=40, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Özellik başına aykırı değer sayısı')
ax.set_ylabel('Özellik adedi')
ax.set_title('Özellik Bazında Aykırı Değer Dağılımı')
plt.tight_layout()
plt.savefig('../figures/fig_aykiri_deger.png', dpi=150, bbox_inches='tight')
plt.show()

---
## EDA Özet

In [ ]:
print('=' * 55)
print('EDA ÖZET TABLOSU')
print('=' * 55)
print(f'Toplam özellik                  : {len(feature_cols)}')
print(f'İstatistiksel anlamlı           : {len(anlamli)} özellik')
print(f'Yüksek korelasyonlu (>0.95)     : {len(to_drop)} özellik elenecek → ~{len(feature_cols)-len(to_drop)} kalır')
print(f'Düşük varyanslı (≤0.01)         : {(variances<=0.01).sum()} özellik elenecek')
print(f'Aykırı değer oranı              : %{satir_bazli/len(df)*100:.1f} (RobustScaler çözüyor)')
print(f'PC1+PC2 açıklanan varyans       : {var1+var2:.1f}%')
print(f'%80 varyans için gereken PC sayısı: {n80}')
print('=' * 55)
print('\nSıradaki adım → 03_on_isleme.ipynb')